# CRBAについて

ロボットの運動方程式を以下とする。

$$
M(\boldsymbol{q}) \ddot{\boldsymbol{q}} + h (\boldsymbol{q}, \dot{\boldsymbol{q}}) = \boldsymbol{\tau}
$$

- $M(\boldsymbol{q})$ : 質量行列 ($n \times n$)
- $h (\boldsymbol{q}, \dot{\boldsymbol{q}})$ : コリオリ・遠心力、重力などのバイアストルク
- $\boldsymbol{\tau}$ : 関節トルク

CRBAは質量行列 $M(\boldsymbol{q})$ を効率的に計算するアルゴリズムである。$h(\boldsymbol{q}, \dot{\boldsymbol{q}})$は計算しない。

### RNEA, ABA , CRBAの違い

|アルゴリズム|入力|出力|
|:---|:---|:---|
|RNEA| $\boldsymbol{q}, \dot{\boldsymbol{q}}, \ddot{\boldsymbol{q}}$ | $\boldsymbol{\tau}$ |
|ABA| $\boldsymbol{q}, \dot{\boldsymbol{q}}, \boldsymbol{\tau}$ | $\ddot{\boldsymbol{q}}$ |
|CRBA| $\boldsymbol{q}$ | $M(\boldsymbol{q})$ |



## 1. 単位ベクトル法

RNEAを使っても質量行列を得ることはできる。

例えば、運動方程式上で以下のように$h$を消去する操作を行う。

$$
\dot{\boldsymbol{q}} = 0, \quad \boldsymbol{g} = 0 \quad \rightarrow \quad h(\boldsymbol{q}, \dot{\boldsymbol{q}}) = 0
$$

すると、以下の式となる。

$$
\tau = M(\boldsymbol{q})\ddot{\boldsymbol{q}}
$$

この式を展開すると例えば以下のようなものである。

$$
\begin{bmatrix}
\tau_1 \\ \tau_2
\end{bmatrix} = 
\begin{bmatrix}
m_{11}(q_1,q_2) & m_{12}(q_1,q_2) \\
m_{21}(q_1,q_2) & m_{22}(q_1,q_2)
\end{bmatrix}
\begin{bmatrix}
\ddot{q}_1 \\ \ddot{q_2}
\end{bmatrix}
$$

各関節を順に単位加速度で動かすことを考える。$e_j$は第$j$成分だけが1の単位ベクトルである。

$$
\ddot{\boldsymbol{q}} = e_j
$$

すると、上記の例に当てはめると、

- $e_1$

$$
\begin{bmatrix}
m_{11}(q_1,q_2) \\ m_{21}(q_1,q_2)
\end{bmatrix} = 
\begin{bmatrix}
m_{11}(q_1,q_2) & m_{12}(q_1,q_2) \\
m_{21}(q_1,q_2) & m_{22}(q_1,q_2)
\end{bmatrix}
\begin{bmatrix}
1 \\ 0
\end{bmatrix}
$$

- $e_2$

$$
\begin{bmatrix}
m_{12}(q_1,q_2) \\ m_{22}(q_1,q_2)
\end{bmatrix} = 
\begin{bmatrix}
m_{11}(q_1,q_2) & m_{12}(q_1,q_2) \\
m_{21}(q_1,q_2) & m_{22}(q_1,q_2)
\end{bmatrix}
\begin{bmatrix}
0 \\ 1
\end{bmatrix}
$$

RNEAは$\boldsymbol{q}, \dot{\boldsymbol{q}}, \ddot{\boldsymbol{q}}$ から$\boldsymbol{\tau}$を効率的に求める方法である。重力を含めると以下の関係である。

$$
\boldsymbol{\tau} = \text{RNEA}(\boldsymbol{q}, \dot{\boldsymbol{q}}, \ddot{\boldsymbol{q}}, \boldsymbol{g})
$$

よって、RNEAを以下のように単位加速度ベクトル$e_j$で計算することを$n$回行えば質量行列全体を求めることが出来る。

$$
M_{:j} = \text{RNEA}(\boldsymbol{q}, 0, e_j, \boldsymbol{g}=0)
$$

一方、CRBAはComposite Rigid Body Inertiaを利用して、より効率的に質量行列を計算する。

## 2. Composite Rigid Body Inertia について

例えば次のような2 link のアームを考える。$q_1,q_2$は回転の正方向を示している。

<p align="center">
<img src="./images/2link_images.png" style="width:50%;" />
</p>

各リンクの[Spatial Inertia](../../ABA/Basic/spatial_vector.ipynb) を以下とする。

$$
{^1}\mathcal{I}_1, \quad {^2}\mathcal{I}_2
$$

剛体の運動エネルギー $T$ はSpatial VelocityとSpatial Inertiaを用いて以下で表される。運動エネルギーは座標系や基準点の取り方によらず不変であるため、ここではSpatial Velocity と Spatial Inertia の座標系と基準の点の表記を省略する。

$$
T = \frac{1}{2} \boldsymbol{V}^T \mathcal{I} \boldsymbol{V}
$$

### 2.1 $J_2$を固定した場合

$J_2$を固定した状態とすると、$J_1$を動作させるとLink1とLink2は一体となって動作する。$J_1$に関するLink1の Spatial Velocity を子リンクLink2の座標・基準点への変換は以下で表す。ここで右下の数字はどのリンクで発生したSpatial Velocityであるかを示している。

$$
{^2} \boldsymbol{V}_1 = {^2}X_1 {^1} \boldsymbol{V}_1
$$

- ${^2}X_1$ : Link1座標でLink1原点を基準点とした Spatial Velocityを、Link2座標でLink2原点を基準としてSpatial Velocityへ変換する行列である。

Link2の運動エネルギー$T_2$は以下となる。

$$
T_2 = \frac{1}{2} \left({^2}\boldsymbol{V}_1\right)^T \ {^2}\mathcal{I}_2 \ {^2}\boldsymbol{V}_{1}
$$

これにSpatial Velocityの変換式を代入すると以下となる。

$$
\begin{aligned}
T_2 &= \frac{1}{2} \left( {^2}X_1 {^1} \boldsymbol{V}_1 \right)^T \ {^2}\mathcal{I}_2 \ \left( {^2}X_1 {^1} \boldsymbol{V}_1 \right)\\
&= \frac{1}{2} {{^1}\boldsymbol{V}_1}^T \left( \left({^2}X_1 \right)^T \ {^2}\mathcal{I}_2 \ {^2}X_1 \right) {^1} \boldsymbol{V}_1\\
\end{aligned}
$$

以下のSpatial Inertia ${^1}\mathcal{I}_2$ を定義する。これは、Link2の慣性をLink1座標系でLink1原点基準に変換したものである。

$$
{^1}\mathcal{I}_2 = \left({{^2}X_1}\right)^T \ {^2}\mathcal{I}_2 \ {^2}X_1
$$

すると、Link2の運動エネルギー$T_2$をLink1のSpatial Velocity ${^1}\boldsymbol{V}_1$ を用いて計算できる。<br>
Link1の運動エネルギー$T_1$は以下である。

$$
T_1 = \frac{1}{2} \left({^1}\boldsymbol{V}_1 \right)^T \ {^1}\mathcal{I}_1 {^1}\boldsymbol{V}_1
$$ 

これより、$J_2$を固定( $\dot{q}_2=\ddot{q}_2=0$ )して$J_1$を動作させたときの、Link1とLink2の全体のエネルギー$T$は以下となる。

$$
\begin{aligned}
T &= T_1 + T_2 \\
&= \frac{1}{2} {{^1}\boldsymbol{V}_1}^T \left( {^1}\mathcal{I}_1 + \left({^2}X_1\right)^T \ {^2}\mathcal{I}_2 \ {^2}X_1\right) {{^1}\boldsymbol{V}_1}
\end{aligned}
$$

以下の定義する。これをLink1をベースとする部分機構のComposite Rigid Body Inertia と呼ぶ。

$$
\boxed{
{^1}\mathcal{I}_1^C = {^1}\mathcal{I}_1 + \left({^2}X_{1}\right)^T {^2}\mathcal{I}_2 {^2}X_{1}
}
$$

これより、運動エネルギー$T$は以下で表すことが出来る。

$$
T = \frac{1}{2} {{^1}\boldsymbol{V}_1}^T {^1}\mathcal{I}_{1}^C {{^1}\boldsymbol{V}_1}
$$ 

### 2.2 $J_2$も動作させる場合

[RNEA](../../ABA/Basic/Spatial-RNEA.ipynb)で定義したSpatial Subspaceを用いたSpatial Velocityの再帰式を用いて、Link2の速度は以下となる。

$$
{^2} \boldsymbol{V}_2 = {^2}X_1 {^1} \boldsymbol{V}_1 + {^2}\boldsymbol{S}_2 \dot{q}_2
$$

これによりLink2の運動エネルギー$T_2$は以下となる。

$$
T_2 = \frac{1}{2} \left({^2}\boldsymbol{V}_2\right)^T \ {^2}\mathcal{I}_2 \ {^2}\boldsymbol{V}_{2}
$$

${^2} \boldsymbol{V}_2$を代入すると、$T_2$は以下である。

$$
\begin{aligned}
T_2 &= \frac{1}{2} \left({^2}X_1 {^1}\boldsymbol{V}_1 + {^2}\boldsymbol{S}_2 \dot{q}_2\right)^T \ {^2}\mathcal{I}_2 \ \left( {^2}X_1 {^1}\boldsymbol{V}_1 + {^2}\boldsymbol{S}_2 \dot{q}_2 \right) \\
&= \frac{1}{2} {{^1}\boldsymbol{V}_1}^T \left({^2}X_1 \right)^T {^2}\mathcal{I}_2 {^2}X_1 {^1}\boldsymbol{V}_1 + \frac{1}{2} {{^1}\boldsymbol{V}_1}^T \left({^2}X_1 \right)^T {^2}\mathcal{I}_2 {^2}\boldsymbol{S}_2 \dot{q}_2 \\
& + \frac{1}{2} \dot{q}_2 {{^2}\boldsymbol{S}_2}^T {^2}\mathcal{I}_2 {^2}X_1 {^1}\boldsymbol{V}_1 + \frac{1}{2} \dot{q}_2 {{^2}\boldsymbol{S}_2}^T {^2}\mathcal{I}_2 {^2}\boldsymbol{S}_2 \dot{q}_2
\end{aligned}
$$

各Spatial Velocityは右下の数字と左上の表す座標数字が同じである省略し、変換行列について座標系を示す左上と右下の数字は変化しないため、こちらの数字も省略する。そして、運動エネルギーはスカラーであるため各項はスカラーとなっている。第3項を転置すると、Spatial Inertiaの対称性より、以下のように2項と一致する。

$$
\frac{1}{2} \boldsymbol{V}_1^T X^T \mathcal{I}_2 {\boldsymbol{S}_2} \dot{q}_2
$$

また、Spatial Subspace を用いて$\boldsymbol{V}_1$ は以下となる。

$$
\boldsymbol{V}_1 = \boldsymbol{S}_1 \dot{q}_1
$$

よって、$T_2$は以下で表現できる。

$$
\begin{aligned}
T_2 &= \frac{1}{2} \dot{q}_1{\boldsymbol{S}_1}^T X^T \mathcal{I}_2 X \boldsymbol{S}_1 \dot{q}_1 +\dot{q}_1 {\boldsymbol{S}_1}^T X^T \mathcal{I}_2 \boldsymbol{S}_2 \dot{q}_2 + \frac{1}{2} \dot{q}_2 {\boldsymbol{S}_2}^T \mathcal{I}_2 \boldsymbol{S}_2 \dot{q}_2
\end{aligned}
$$

Link1の運動エネルギー$T_1$は以下である。

$$
T_1 = \frac{1}{2} \dot{q}_1 \boldsymbol{S}_1^T \mathcal{I}_1 \boldsymbol{S}_1 \dot{q}_1
$$

よって、全運動エネルギー$T$は以下となる。

$$
\begin{aligned}
T &= T_1 + T_2 \\
&=  \frac{1}{2} \dot{q}_1 \boldsymbol{S}_1^T \left(\mathcal{I}_1 + X^T \mathcal{I}_2 X\right) \boldsymbol{S}_1 \dot{q}_1 \\ 
&\quad +\dot{q}_1 {\boldsymbol{S}_1}^T X^T \mathcal{I}_2 \boldsymbol{S}_2 \dot{q}_2 + \frac{1}{2} \dot{q}_2 {\boldsymbol{S}_2}^T \mathcal{I}_2 \boldsymbol{S}_2 \dot{q}_2
\end{aligned}
$$

2.1節のComposite Rigid Body Inertia $\mathcal{I}_1^C =\mathcal{I}_1 + X^T \mathcal{I}_2 X$を用いると、以下で表現できる。

$$
T = \frac{1}{2} \dot{q}_1 \boldsymbol{S}_1^T \mathcal{I}_1^C \boldsymbol{S}_1 \dot{q}_1 +\dot{q}_1 {\boldsymbol{S}_1}^T X^T \mathcal{I}_2 \boldsymbol{S}_2 \dot{q}_2 + \frac{1}{2} \dot{q}_2 {\boldsymbol{S}_2}^T \mathcal{I}_2 \boldsymbol{S}_2 \dot{q}_2
$$

一方、一般的な2自由度系の運動エネルギーは以下のようになる。

$$
T = \frac{1}{2}\begin{bmatrix}\dot{q}_1 & \dot{q}_2 \end{bmatrix}
\begin{bmatrix} M_{11} & M_{12} \\ M_{21} & M_{22} \end{bmatrix}
\begin{bmatrix}\dot{q}_1 \\ \dot{q}_2 \end{bmatrix}
$$

慣性行列の対称性$M_{12} = M_{21}$を用いて展開すると、以下となる。

$$
T = \frac{1}{2} M_{11} {\dot{q}_1}^2 + M_{12} \dot{q}_1 \dot{q}_2 + \frac{1}{2} M_{22} {\dot{q}_2}^2
$$

これより、係数を比較すれば以下のlink2も動作させた場合の質量慣性行列$M(\boldsymbol{q})$の各要素が現れる。

$$
\boxed{
\begin{aligned}
M_{11} &= \boldsymbol{S}_1^T \mathcal{I}_1^C \boldsymbol{S}_1 \\
M_{12} &= {\boldsymbol{S}_1}^T X^T \mathcal{I}_2 \boldsymbol{S}_2 \\
M_{21} &= M_{12} \\
M_{22} &= {\boldsymbol{S}_2}^T \mathcal{I}_2 \boldsymbol{S}_2
\end{aligned}
}
$$

各項の意味は次となる。

- $M_{11}$ : $\dot{q}_2=0$として$J_1$を動かすときの、Link1とLink2の合成慣性(Composite Rigid Body Inertia)を$J_1$軸へ射影した値
- $M_{22}$ : $\dot{q}_1=0$として$J_2$を動かすときの、Link2の慣性を$J_2$軸へ射影した値
- $M_{12} = M_{21}$ : $J_1$と$J_2$の運動の結合慣性を表す成分





## 3. Composite Rigid Body Inertia の再帰式の導出

### 3.1 3リンクアームの場合

対象を3linkアームとし、リンク数を拡大した場合から、Composite Rigid Body Inertia の再帰式を導出する。

<p align="center">
<img src="./images/3link_images.png" style="width:55%;" />
</p>

各リンクのSpatial Inertiaを以下とする。

$$
{^1}\mathcal{I}_1, \quad {^2}\mathcal{I}_2, \quad {^3}\mathcal{I}_3
$$

これより、各リンクを根とする部分機構の Composite Rigid Body Inertia (CRBI) を考える。

- Link3 のCRBI

    Link3 には子リンクがない。これより、Link3を根とするCRBIは以下となる。

    $$
    {^3}\mathcal{I}_3^C = {^3}\mathcal{I}_3
    $$

- Link2 のCRBI

    Link2を根とする部分機構は、Link2とLink3となる。 第2節で導出した式を利用すると、Link2を根とするCRBIは以下となる。

    $$
    {^2}\mathcal{I}_2^C = {^2}\mathcal{I}_2 + \left( {^3}X{_2} \right)^T \ {^3}\mathcal{I}{_3}^C \ {^3}X{_2}
    $$ 

    これは、Link3のSpatial InertiaをLink2座標系・Link2原点基準に変換して足している。

- Link1のCRBI

    Link1を根とする部分機構は、Link1、Link2とLink3となる。Link2とLink3の慣性は、${^2}\mathcal{I}_2^C$ にまとめられている。よって、Link1を根とするCRBIは以下となる。

    $$
    {^1}\mathcal{I}{_1}^C = {^1}\mathcal{I}{_1} + \left({^2}X{_1} \right)^T \ {^2}\mathcal{I}{_2}^C \ {^2}X{_1}
    $$

### 3.2 $n$リンクへの一般化

リンク$i$の子リンクの集合をchildren($i$)とする。すると、Composite Rigid Body Inertia は以下で表現される。

$$
{^i}\mathcal{I}{_i}^C = {^i}\mathcal{I}{_i} + \sum_{j \in \text{children}(i)} \left({^j}X{_i} \right)^T \ {^j}\mathcal{I}{_j}^C \ {^j}X{_i}
$$

- ${^i}\mathcal{I}{_i}$ : Link i 自身のSpatial Inertia
- ${^i}\mathcal{I}{_i}^C$ : リンク $i$ を根とする部分機構のCRBI
- ${^j}X{_i}$ : 親リンクiから子リンクjへSpatial Motion Transform

この子リンクの集合とは、以下のようなリンク構造のLink2にとってのLink3とLink4のことである。

<p align="center">
<img src="./images/n_link_images.png" style="width:50%;"/>
</p>

この場合、

$$
\text{children}(2) = \{3,4\}
$$

となり、Link2を根とする部分機構のCRBIは以下となる。

$$
{^2}\mathcal{I}{_2}^C = {^2}\mathcal{I}{_2} + \left({^3}X{_2} \right)^T \ {^3}\mathcal{I}{_3}^C \ {^3}X{_2} + \left({^4}X{_2} \right)^T \ {^4}\mathcal{I}{_4}^C \ {^4}X{_2} 
$$

そして、Link1では以下となる。

$$
{^1}\mathcal{I}{_1}^C = {^1}\mathcal{I}{_1} + \left({^2}X{_1} \right)^T \ {^2}\mathcal{I}{_2}^C \ {^2}X{_1}
$$


これより、3.1節でみたように、末端リンクからベースに向かって順に計算すれば、全てのCRBIを求めることが出来る。このBackward Recursionでは末端リンクから根元へSpatial Inertia を合成し、各リンクを根とする部分機構の Composite Rigid Body Inertia を計算する。これらをMotion Subspaceへ射影することで、質量行列の対角要素を求められる。また、非対角成分の計算にも利用できる。


### 3.3 Composite Rigid Body Inertia から質量行列を計算する

#### 3.3.1 2 link arm の場合

2.2節で求めた2 link armのCRBIを用いた質量行列の各成分は以下であった。

$$
\begin{aligned}
M_{11} &= \boldsymbol{S}_1^T \mathcal{I}_1^C \boldsymbol{S}_1 \\
M_{12} &= {\boldsymbol{S}_1}^T \left({^2}X{_1}\right)^T \mathcal{I}_2 \boldsymbol{S}_2 \\
M_{21} &= M_{12} \\
M_{22} &= {\boldsymbol{S}_2}^T \mathcal{I}_2 \boldsymbol{S}_2
\end{aligned}
$$

非対角要素の$M_{12}$は $\ddot{q}_2$ の係数となるため、以下のような関係式となる。

$$
{\tau_1}^{(2)} = {\boldsymbol{S}_1}^T \left({^2}X{_1}\right)^T \mathcal{I}_2 \boldsymbol{S}_2 \ddot{q}_2
$$

ここで、[RNEA](../../ABA/Basic/Spatial-RNEA.ipynb) で用いた Spatial Acceleration $\boldsymbol{A}_2$ の再帰式は以下であった。

$$
\boldsymbol{A}_2 = {^2}X{_1} \boldsymbol{A}_1 + \boldsymbol{S}_2 \ddot{q}_2 + \text{crm}(\boldsymbol{V}_2) \boldsymbol{S}_2 \dot{q}_2
$$

ここから $\ddot{q}_2$ に関する項だけを抽出すると以下となる。

$$
{\boldsymbol{A}_2}^{(2)} = \boldsymbol{S}_2 \ddot{q}_2
$$

また、[Spatial Newton-Euler方程式](../../ABA/Basic/Spatial_Newton_Euler.ipynb) は以下である。

$$
\boldsymbol{f}_2 = \mathcal{I}_2 \boldsymbol{A}_2 + \text{crf}(\boldsymbol{V}_2) \mathcal{I}_2 \boldsymbol{V}_2
$$

これに $\ddot{q}_2$ に比例する項だけ抽出すると以下となる。

$$
{\bar{\boldsymbol{f}}_2}^{(2)} = \mathcal{I}_2 \boldsymbol{S}_2 \ddot{q}_2
$$

そして、$\ddot{q}_2=1$と単位角加速度とすれば以下となる。これは右上の$(2)$で$\ddot{q}_2$の単位加速度に対する係数であることを示している。

$$
{{\boldsymbol{f}}_2}^{(2)} = \mathcal{I}_2 \boldsymbol{S}_2
$$

$M_{12} = {\boldsymbol{S}_1}^T \left({^2}X{_1}\right)^T \mathcal{I}_2 \boldsymbol{S}_2$ に対して適用すると

$$
\begin{aligned}
M_{12} &= {\boldsymbol{S}_1}^T \left({^2}X{_1}\right)^T {{\boldsymbol{f}}_2}^{(2)} \\
\end{aligned}
$$

となり、変換行列により、

$$
{^1}{{\boldsymbol{f}}_2}^{(2)} = \left({^2}X{_1}\right)^T {^2}{{\boldsymbol{f}}_2}^{(2)}
$$ 

よって、

$$
\begin{aligned}
M_{12} &= {\boldsymbol{S}_1}^T {^1}{{\boldsymbol{f}}_2}^{(2)} \\
\end{aligned}
$$

となる。

また、${{\boldsymbol{f}}_2}^{(2)}$ を $\boldsymbol{S}_2$ を使ってLink2 の変換すると、$M_{22}$となる。

$$
M_{22} = {\boldsymbol{S}_2}^T \ {{\boldsymbol{f}}_2}^{(2)} =  {\boldsymbol{S}_2}^T \ \mathcal{I}_2 \ \boldsymbol{S}_2
$$

考え方をまとめると、末端リンクから計算を行うと、

- $\ddot{q}_2$ の単位角加速度によるSpatial Forceの係数を計算する。

    $$
    {{\boldsymbol{f}}_2}^{(2)} = \mathcal{I}_2 \boldsymbol{S}_2
    $$

- ${{\boldsymbol{f}}_2}^{(2)}$ を Link2の Subspace $\boldsymbol{S}_2$ で$J_2$へ射影すると、$M_{22}$ が求まる。

    $$
    M_{22} = {\boldsymbol{S}_2}^T \ {{\boldsymbol{f}}_2}^{(2)} = {\boldsymbol{S}_2}^T \ \mathcal{I}_2 \boldsymbol{S}_2
    $$

- ${{\boldsymbol{f}}_2}^{(2)}$ を Link1へ変換し、Subspace  $\boldsymbol{S}_1$ で$J_1$で射影すると、$M_{12}$が求まる。 

    $$
    M_{12} = {\boldsymbol{S}_1}^T \ ({^2}X{_1})^T \ {{\boldsymbol{f}}_2}^{(2)} = {\boldsymbol{S}_1}^T \ ({^2}X{_1})^T \mathcal{I}_2 \boldsymbol{S}_2
    $$

- $\ddot{q}_1$ の単位角加速度によるSpatial Forceの係数を計算する。この場合Link2も一緒に動くため、Link1を根とするCRBIとなる。

    $$
    {{\boldsymbol{f}}_1}^{(1)} = \mathcal{I}_1^C \boldsymbol{S}_1
    $$

- ${{\boldsymbol{f}}_1}^{(1)}$ を Link1の Subspace $\boldsymbol{S}_1$ で$J_1$へ射影すると、$M_{11}$ が求まる。

    $$
    M_{11} = {\boldsymbol{S}_1}^T \ {{\boldsymbol{f}}_1}^{(1)} = {\boldsymbol{S}_1}^T \mathcal{I}_1^C \boldsymbol{S}_1
    $$


この考え方はリンク数が増えると次のようになる。

#### 3.3.2. 3 link arm の場合

##### CRBIが質量行列の要素になる理由

3 Link armの質量行列は以下のような形になる。

$$
M(\boldsymbol{q}) = \begin{bmatrix}
M_{11} & M_{12} & M_{13} \\
M_{21} & M_{22} & M_{23} \\
M_{31} & M_{32} & M_{33}
\end{bmatrix}
$$

非対角要素$M_{12}$について考える。これは、$\tau_1$に関する$\ddot{q}_2$の項の係数である。

よって、Link2 の加速度に関して抽出すると以下となる。

$$
\boldsymbol{A}_2^{(2)} = \boldsymbol{S}_2 \ddot{q}_2
$$

Link3 には 親リンクのLink2 から加速度が伝わる。

$$
\boldsymbol{A}_3^{(2)} = {^3}X{_2} \boldsymbol{S}_2 \ddot{q}_2
$$

したがってLink2とLink3の慣性Spatial Force をLink2 座標系にまとめると、

$$
\begin{aligned}
{\boldsymbol{f}_2}^{(2)} &= \mathcal{I}_2 \boldsymbol{S}_2 \ddot{q}_2 + ({^3}X{_2})^T \ \mathcal{I}_3 {^3}X{_2} \boldsymbol{S}_2 \ddot{q}_2 \\
&= \left( \mathcal{I}_2 + ({^3}X{_2})^T \ \mathcal{I}_3 {^3}X{_2} \right) \boldsymbol{S}_2 \ddot{q}_2 \\
&= \mathcal{I}_2^{C} \boldsymbol{S}_2 \ddot{q}_2 
\end{aligned}
$$

これより、Link2 を単位角加速度で動作させるとしてもLink3が一緒に動き、その時にはLink2を根とするCRBIが出てくる。

$M_{12}$ は$\tau_1$への寄与となるため、${\boldsymbol{f}_2}^{(2)}$ をLink1へ変換を行う。

$$
{\boldsymbol{f}_1}^{(2)} = ({^2}X{_1})^T \ \mathcal{I}_2^{C} \ \boldsymbol{S}_2 \ddot{q}_2 
$$

これをLink1のMotion Subspace $\boldsymbol{S}_1$ で射影すると、$\ddot{q}_2$ の項が寄与したLink1へのトルクになる。

$$
\tau_{1}^{(2)} = \boldsymbol{S}_1^T {\boldsymbol{f}_1}^{(2)} = \boldsymbol{S}_1^T ({^2}X{_1})^T \ \mathcal{I}_2^{C} \ \boldsymbol{S}_2 \ddot{q}_2
$$

ここから係数を取り出すと、$M_{12}$となる。

$$
M_{12} = \boldsymbol{S}_1^T ({^2}X{_1})^T \ \mathcal{I}_2^{C} \ \boldsymbol{S}_2
$$


##### 質量行列の計算

CRBIが以下のように求まっているとする。

$$
{^1}\mathcal{I}{_1}^C, \quad {^2}\mathcal{I}{_2}^C, \quad {^3}\mathcal{I}{_3}^C
$$ 

- 1. 末端リンクの単位角加速度の係数の計算

    $$
    {\boldsymbol{f}_3}^{(3)} = {^3}\mathcal{I}{_3}^C \boldsymbol{S}_3
    $$

- 2. $M_{33}$の計算 : ${\boldsymbol{f}_3}^{(3)}$ を $\boldsymbol{S}_3$ で $J_3$ へ射影する。

    $$
    M_{33} = {\boldsymbol{S}_3}^T {\boldsymbol{f}_3}^{(3)}
    $$

- 3. ${\boldsymbol{f}_3}^{(3)}$ を Link2へ変換して単位角加速度の係数を計算する。

    $$
    {\boldsymbol{f}_2}^{(3)} = ({^3}X{_2})^T \ {\boldsymbol{f}_3}^{(3)}
    $$

- 4. $M_{23}$の計算 : ${\boldsymbol{f}_2}^{(3)}$ を $\boldsymbol{S}_2$ で $J_2$ へ射影する。

    $$
    M_{23} = {\boldsymbol{S}_2}^T {\boldsymbol{f}_2}^{(3)}
    $$

- 5. ${\boldsymbol{f}_2}^{(3)}$ を Link1へ変換して単位角加速度の係数を計算する。

    $$
    {\boldsymbol{f}_1}^{(3)} = ({^2}X{_1})^T \ {\boldsymbol{f}_2}^{(3)}
    $$

- 6. $M_{13}$の計算 : ${\boldsymbol{f}_1}^{(3)}$ を $\boldsymbol{S}_1$ で $J_1$ へ射影する。

    $$
    M_{13} = {\boldsymbol{S}_1}^T {\boldsymbol{f}_1}^{(3)}
    $$
 
 - 7. Link2の単位角加速度の係数の計算

    $$
    {\boldsymbol{f}_2}^{(2)} = {^2}\mathcal{I}{_2}^C \boldsymbol{S}_2
    $$

- 8. $M_{22}$の計算 : ${\boldsymbol{f}_2}^{(2)}$ を $\boldsymbol{S}_2$ で $J_2$ へ射影する。

    $$
    M_{22} = {\boldsymbol{S}_2}^T {\boldsymbol{f}_2}^{(2)}
    $$

- 9. ${\boldsymbol{f}_2}^{(2)}$ を Link1へ変換して単位角加速度の係数を計算する。

    $$
    {\boldsymbol{f}_1}^{(2)} = ({^2}X{_1})^T \ {\boldsymbol{f}_2}^{(2)}
    $$

- 10. $M_{12}$の計算 : ${\boldsymbol{f}_1}^{(2)}$ を $\boldsymbol{S}_1$ で $J_1$ へ射影する。

    $$
    M_{12} = {\boldsymbol{S}_1}^T {\boldsymbol{f}_1}^{(2)}
    $$

- 11. Link1の単位角加速度の係数の計算

    $$
    {\boldsymbol{f}_1}^{(1)} = {^1}\mathcal{I}{_1}^C \boldsymbol{S}_1
    $$

- 12. ${\boldsymbol{f}_1}^{(1)}$ を $\boldsymbol{S}_1$ で $J_1$ へ射影する。

    $$
    M_{11} = {\boldsymbol{S}_1}^T {\boldsymbol{f}_1}^{(1)}
    $$


これで質量行列の上三角行列の要素が計算された。下三角の非対角項は上三角の非対角項と一緒であるため計算の必要はない。

よって質量行列$M(\boldsymbol{q})$を構成できる。

$$
M(\boldsymbol{q}) = \begin{bmatrix}
M_{11} & M_{12} & M_{13} \\
M_{21} & M_{22} & M_{23} \\
M_{31} & M_{32} & M_{33}
\end{bmatrix}
$$


## 4. CRBAのアルゴリズム

3節でCRBIを末端リンクからベースに向かって計算する再帰式と、そのCRBIを用いた質量行列の要素を計算する流れを、3リンクアームで具体的な式で計算した。この節では Composite Rigid Body Algorithm としてより一般的な記述としてまとめる。

アームはnリンクとして、途中で子リンクの分岐はないストレートなタイプとする。

### 4.1 事前計算

- 変換行列

    Spatial Velocity のSpatial Transform を、座標を親リンク i-1 から子リンク i へ移し、基準点の親リンクの原点$O_{i-1}$から子リンクの原点$O_{i}$へ変換する行列を ${}^{i}X{_{i-1}}$ として表す。

    $$
    {}^{i}\boldsymbol{V}_{i-1, O_{i}} = {}^{i}X{_{i-1}} \ {}^{i-1}\boldsymbol{V}_{{i-1}, O_{i-1}}
    $$

    $$
    {}^{i}X{_{i-1}} = \begin{bmatrix}
    {}^{i} R_{i-1} & 0 \\
    - \left[{}^{i} \boldsymbol{r}_{O_{i-1} \ O_{i}}\right]^\land {}^{i} R_{i-1} & {}^{i} R_{i-1}
    \end{bmatrix}
    $$


    この変換行列を n-1 個準備する。
    

- 各リンクのSpatial Inertia

    それぞれのリンク座標でリンク原点を基準点とするSpatial Inertia を求める。

    $$
    {}^{i} \mathcal{I}{_i} = \begin{bmatrix}
    {}^{i}\boldsymbol{I}_{G_i} - m_i [{}^{i}\boldsymbol{r}_{O_i G_i}]^\land [{}^{i}\boldsymbol{r}_{O_i G_i}]^\land & m_i [{}^{i}\boldsymbol{r}_{O_i G_i}]^\land \\
    - m_i [{}^{i}\boldsymbol{r}_{O_i G_i}]^\land & m_i \boldsymbol{I}_3 
    \end{bmatrix}
    $$

    - ${}^{i}\boldsymbol{I}_{G_i}$ : Link i の重心における慣性テンソル
    - $m_i$ : Link i の質量
    - ${}^{i}\boldsymbol{r}_{O_i G_i}$ : Link i の原点$O_i$からLink i の重心 $G_i$ までの位置ベクトル
    
    この Spatial Inertia を n 個準備する。

### 4.2 Pass 1 : Composite Rigid Body Inertia の計算 

末端リンク n のCRBI を設定後、ベースに向かってLink1まで以下の式を再帰式を計算する。

$$
\begin{aligned}
{{}^{n} \mathcal{I}{_n}}^{C} &= {}^{n} \mathcal{I}{_n} \\
{{}^{i-1} \mathcal{I}{}_{i-1}}^{C} &= {}^{i-1} \mathcal{I}{_{i-1}} + ({}^{i}X{_{i-1}})^T \ {{}^{i} \mathcal{I}{_{i}}}^{C} \ {}^{i}X{_{i-1}} \quad (i = n , ... , 2)
\end{aligned}
$$

### 4.3 Pass 2 : 質量行列の計算

質量行列の対角成分について、$i = n, n-1, ..., 1$ として各リンクi について次の計算で求める。

$$
\begin{aligned}
\boldsymbol{f}{_i}^{(i)} &= {{}^{i}\mathcal{I}{_i}}^C \boldsymbol{S}{_i} \\
M_{ii} &= \boldsymbol{S}{_i}^T \ \boldsymbol{f}{_i}^{(i)}
\end{aligned}
$$

次に非対角成分について、$k=i,i-1,...,2$ として、次の計算の再帰計算で求める。

$$
\begin{aligned}
\boldsymbol{f}{_{k-1}}^{(i)} &= (^k X_{k-1})^T \boldsymbol{f}{_k}^{(i)} \\
M_{k-1,i} &= \boldsymbol{S}{_{k-1}}^T \ \boldsymbol{f}{_{k-1}}^{(i)} \\
M_{i, k-1} &= M_{k-1, i}
\end{aligned}
$$



## 5. 2 Link Arm を対象とした検証

[このNote](../../3D-EoM/2LinkArm/Kane.ipynb)では、2Link Arm の運動方程式を以下のように導いた。

$$
\begin{bmatrix}
I_{1zz} +  S_{q_2}^2 (I_{2xx} + m_2r_{2G}^2) + C_{q_2}^2 I_{2zz} & 0 \\
0 & m_2r_{2G}^2 + I_{2yy}
\end{bmatrix}
\begin{bmatrix}
\ddot{q}_1 \\ \ddot{q}_2 
\end{bmatrix} +
\begin{bmatrix}
2S_{q_2}C_{q_2} \left(m_2r_{2G}^2 + I_{2xx} - I_{2zz} \right)\dot{q}_1\dot{q}_2 \\
 S_{q_2}C_{q_2}(I_{2zz}-I_{2xx} - m_2r_{2G}^2)\dot{q}_1^2
\end{bmatrix} +
\begin{bmatrix}
0 \\
- r_{2G} S_{q_2} m_2 g 
\end{bmatrix}=
\begin{bmatrix}
\tau_1 \\ \tau_2
\end{bmatrix}
$$

ここから、質量行列 $M(\boldsymbol{q})$ は以下である。

$$
\begin{bmatrix}
I_{1zz} +  S_{q_2}^2 (I_{2xx} + m_2r_{2G}^2) + C_{q_2}^2 I_{2zz} & 0 \\
0 & m_2r_{2G}^2 + I_{2yy}
\end{bmatrix}
$$

これをCRBAの方法で計算により求めてみる。

### 5.1 事前設定

以降で用いる記号を定義する。$C_{q_2}$ と $S_{q_2}$ はそれぞれ $\cos(q_2)$ と $\sin(q_2)$、$r_{iG}$ はLink $i$ の原点から重心までの距離、$l_1$ はLink1の長さである。

$$
C_{q_2}=\cos(q_2),\qquad S_{q_2}=\sin(q_2),\qquad
r_{iG}=\frac{L_{z,i}}{2},\qquad l_1=L_{z,1}
$$

#### 5.1.1 Spatial Transform

2リンクアームであるため、変換行列はLink1からLink2への変換が一つある。

$$
{}^{2}X{_{1}} = \begin{bmatrix}
{}^{2} R_{1} & 0 \\
- \left[{}^{2} \boldsymbol{r}_{O_{1} \ O_{2}}\right]^\land {}^{2} R_{1} & {}^{2} R_{1}
\end{bmatrix}
$$

それぞの項を[このNote](../../3D-EoM/2LinkArm/Kane.ipynb)から読み解くと以下となる。

$$
{}^{2} R_{1} = 
\begin{bmatrix}
C_{q_2} & 0 & S_{q_2} \\
0 & 1 & 0 \\
- S_{q_2} & 0 & C_{q_2}\\
\end{bmatrix}
$$

${}^{2} \boldsymbol{r}_{O_{1} \ O_{2}} = {}^{2} R_{1} \ {}^{1} \boldsymbol{r}_{O_{1} \ O_{2}}$ より、

$$
{}^{2} \boldsymbol{r}_{O_{1} \ O_{2}} = \begin{bmatrix}
C_{q_2} & 0 & S_{q_2} \\
0 & 1 & 0 \\
- S_{q_2} & 0 & C_{q_2}\\
\end{bmatrix}
\begin{bmatrix} 0 \\ 0 \\ l_1 \end{bmatrix} = 
\begin{bmatrix} S_{q_2} l_1 \\ 0 \\ C_{q_2} l_1 \end{bmatrix}
$$

ここから、左下の要素は以下となる。

$$
-\left[{}^{2} \boldsymbol{r}_{O_{1} \ O_{2}}\right]^\land {}^{2} R_{1} = -
\begin{bmatrix}
0 & - C_{q_2} l_1 & 0 \\
C_{q_2} l_1 & 0 & - S_{q_2} l_1  \\
0 & S_{q_2} l_1  & 0 \\
\end{bmatrix}
\begin{bmatrix}
C_{q_2} & 0 & S_{q_2} \\
0 & 1 & 0 \\
- S_{q_2} & 0 & C_{q_2}\\
\end{bmatrix} = \begin{bmatrix}
0 & C_{q_2} l_1 & 0 \\
- l_1 & 0 & 0 \\
 0 & - S_{q_2} l_1 & 0 
\end{bmatrix}
$$

よって、

$$
{}^2X_1=
\begin{bmatrix}
C_{q_2}&0&S_{q_2}&0&0&0\\
0&1&0&0&0&0\\
-S_{q_2}&0&C_{q_2}&0&0&0\\
0&C_{q_2}l_1&0&C_{q_2}&0&S_{q_2}\\
-l_1&0&0&0&1&0\\
0&-S_{q_2}l_1&0&-S_{q_2}&0&C_{q_2}
\end{bmatrix}
$$


#### 5.1.2 Spatial Inertia

各リンクのSpatal Inertia は以下である。

$$
{}^{i} \mathcal{I}{_i} = \begin{bmatrix}
{}^{i}\boldsymbol{I}_{G_i} - m_i [{}^{i}\boldsymbol{r}_{O_i G_i}]^\land [{}^{i}\boldsymbol{r}_{O_i G_i}]^\land & m_i [{}^{i}\boldsymbol{r}_{O_i G_i}]^\land \\
- m_i [{}^{i}\boldsymbol{r}_{O_i G_i}]^\land & m_i \boldsymbol{I}_3 
\end{bmatrix}
$$

それぞの項を[このNote](../../3D-EoM/2LinkArm/Kane.ipynb)から読み解くと以下となる。

- Link1 

    リンク質量

    $$
    m_1
    $$

    重心周りの慣性テンソルは以下である。

    $$
    {^1}\boldsymbol{I}_{G_1} = \begin{bmatrix}
    I_{1xx} & 0 & 0 \\
    0 & I_{1yy} & 0 \\
    0 & 0 & I_{1zz}
    \end{bmatrix}
    $$

    リンク原点からリンク重心までの位置ベクトルは以下である。

    $$
    {^1}\boldsymbol{r}_{O_1 G_1} = \begin{bmatrix} 0 \\ 0 \\ r_{1G} \end{bmatrix}
    $$

    これより

    $$
    [{}^{1}\boldsymbol{r}_{O_1 G_1}]^\land = \begin{bmatrix}
    0 & -r_{1G} & 0 \\
    r_{1G} & 0 & 0 \\
    0 & 0 & 0
    \end{bmatrix}
    $$

    また、

    $$
    [{}^{1}\boldsymbol{r}_{O_1 G_1}]^\land \ [{}^{1}\boldsymbol{r}_{O_1 G_1}]^\land = 
    \begin{bmatrix}
    -{r_{1G}}^2 & 0 & 0 \\
    0 & -{r_{1G}}^2 & 0 \\
    0 & 0 & 0
    \end{bmatrix}
    $$

    これより左上の要素は以下となる。
    
    $$
    {}^{1}\boldsymbol{I}_{G_1} - m_1 [{}^{1}\boldsymbol{r}_{O_1 G_1}]^\land [{}^{1}\boldsymbol{r}_{O_1 G_1}]^\land = 
    \begin{bmatrix}
    I_{1xx} + m_1 {r_{1G}}^2 & 0 & 0 \\
    0 & I_{1yy} + m_1 {r_{1G}}^2 & 0 \\
    0 & 0 & I_{1zz}
    \end{bmatrix}
    $$

    よって、

    $$
    {}^1\mathcal{I}_1=
    \begin{bmatrix}
    I_{1xx}+m_1r_{1G}^2&0&0&0&-m_1r_{1G}&0\\
    0&I_{1yy}+m_1r_{1G}^2&0&m_1r_{1G}&0&0\\
    0&0&I_{1zz}&0&0&0\\
    0&m_1r_{1G}&0&m_1&0&0\\
    -m_1r_{1G}&0&0&0&m_1&0\\
    0&0&0&0&0&m_1
    \end{bmatrix}
    $$

- Link2 

    リンク質量

    $$
    m_2
    $$

    重心周りの慣性テンソルは以下である。

    $$
    {^2}\boldsymbol{I}_{G_2} = \begin{bmatrix}
    I_{2xx} & 0 & 0 \\
    0 & I_{2yy} & 0 \\
    0 & 0 & I_{2zz}
    \end{bmatrix}
    $$

    リンク原点からリンク重心までの位置ベクトルは以下である。

    $$
    {^2}\boldsymbol{r}_{O_2 G_2} = \begin{bmatrix} 0 \\ 0 \\ r_{2G} \end{bmatrix}
    $$

    これより

    $$
    [{}^{2}\boldsymbol{r}_{O_2 G_2}]^\land = \begin{bmatrix}
    0 & -r_{2G} & 0 \\
    r_{2G} & 0 & 0 \\
    0 & 0 & 0
    \end{bmatrix}
    $$

    また、

    $$
    [{}^{2}\boldsymbol{r}_{O_2 G_2}]^\land \ [{}^{2}\boldsymbol{r}_{O_2 G_2}]^\land = 
    \begin{bmatrix}
    -{r_{2G}}^2 & 0 & 0 \\
    0 & -{r_{2G}}^2 & 0 \\
    0 & 0 & 0
    \end{bmatrix}
    $$

    これより左上の要素は以下となる。
    
    $$
    {}^{2}\boldsymbol{I}_{G_2} - m_2 [{}^{2}\boldsymbol{r}_{O_2 G_2}]^\land [{}^{2}\boldsymbol{r}_{O_2 G_2}]^\land = 
    \begin{bmatrix}
    I_{2xx} + m_2 {r_{2G}}^2 & 0 & 0 \\
    0 & I_{2yy} + m_2 {r_{2G}}^2 & 0 \\
    0 & 0 & I_{2zz}
    \end{bmatrix}
    $$

    よって、

    $$
    {}^2\mathcal{I}_2=
    \begin{bmatrix}
    I_{2xx}+m_2r_{2G}^2&0&0&0&-m_2r_{2G}&0\\
    0&I_{2yy}+m_2r_{2G}^2&0&m_2r_{2G}&0&0\\
    0&0&I_{2zz}&0&0&0\\
    0&m_2r_{2G}&0&m_2&0&0\\
    -m_2r_{2G}&0&0&0&m_2&0\\
    0&0&0&0&0&m_2
    \end{bmatrix}
    $$



#### 5.1.3 Motion Subspace

再帰計算では 各関節の Motion Subspace も用いる。

- link 1

    $$
    \boldsymbol{S}{_1} = \begin{bmatrix} 0 & 0 & 1 & 0 & 0 & 0 \end{bmatrix}^T
    $$

- link 2
    
    $$
    \boldsymbol{S}{_2} = \begin{bmatrix} 0 & 1 & 0 & 0 & 0 & 0 \end{bmatrix}^T
    $$




### 5.2 CRBA Pass

#### 5.2.1 Pass 1 : CRBIの再帰計算

子リンクの無い末端のLink2からLink1の順に計算する。

$$
\begin{aligned}
{}^2\mathcal{I}_2^C &={}^2\mathcal{I}_2 \\
{}^1\mathcal{I}_1^C &={}^1\mathcal{I}_1+({}^2X_1)^T\,{}^2\mathcal{I}_2^C\,{}^2X_1
\end{aligned}
$$

#### 5.2.2 Pass 2 : 質量行列の各要素の計算

$$
\boldsymbol{f}_2^{(2)} = {^2}\mathcal{I}_{2}^C \boldsymbol{S}_2
$$

$$
M_{22} = \boldsymbol{S}_2^T \ \boldsymbol{f}_2^{(2)}
$$

$$
\boldsymbol{f}_1^{(2)} = ({}^2 X_{1})^T \ \boldsymbol{f}_2^{(2)}
$$

$$
M_{12} = \boldsymbol{S}_1^T \ \boldsymbol{f}_1^{(2)}
$$

$$
M_{21} = M_{12}
$$

$$
\boldsymbol{f}_1^{(1)} = {^1}\mathcal{I}_{1}^C \boldsymbol{S}_1
$$

$$
M_{11} = \boldsymbol{S}_1^T \ \boldsymbol{f}_1^{(1)}
$$



### 5.3 SymPyによる計算

6 x 6 行列を手計算していくことが困難であるため、SymPyによるSymbolic 計算による質量行列の算出を行う。


In [6]:
import sympy as sp

# 1. 記号の定義
q2, l1, r1G, r2G = sp.symbols("q2 l1 r1G r2G", real=True)
m1, m2 = sp.symbols("m1 m2", positive=True)

I1xx, I1yy, I1zz = sp.symbols("I1xx I1yy I1zz", real=True)
I2xx, I2yy, I2zz = sp.symbols("I2xx I2yy I2zz", real=True)

c = sp.cos(q2)
s = sp.sin(q2)

def skew(v):
    return sp.Matrix([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])

def spatial_inertia(m, IG, r):
    rx = skew(r)
    return sp.BlockMatrix([
        [IG - m * rx * rx, m * rx],
        [-m * rx, m * sp.eye(3)]
    ]).as_explicit()

# 2. Spatial Transform
R21 = sp.Matrix([
    [c, 0, s],
    [0, 1, 0],
    [-s, 0, c]
])

r12 = R21 * sp.Matrix([0, 0, l1])

X21 = sp.BlockMatrix([
    [R21, sp.zeros(3)],
    [-skew(r12) * R21, R21]
]).as_explicit()

# 3. 各リンクのSpatial Inertia
IG1 = sp.diag(I1xx, I1yy, I1zz)
IG2 = sp.diag(I2xx, I2yy, I2zz)

I1 = spatial_inertia(m1, IG1, sp.Matrix([0, 0, r1G]))
I2 = spatial_inertia(m2, IG2, sp.Matrix([0, 0, r2G]))

# 4. Motion Subspace
S1 = sp.Matrix([0, 0, 1, 0, 0, 0])
S2 = sp.Matrix([0, 1, 0, 0, 0, 0])

# 5. CRBA Pass 1
IC2 = I2
IC1 = sp.simplify(I1 + X21.T * IC2 * X21)

# 6. CRBA Pass 2
f2 = IC2 * S2
M22 = sp.trigsimp((S2.T * f2)[0])

f1_from_2 = X21.T * f2
M12 = sp.trigsimp((S1.T * f1_from_2)[0])

f1 = IC1 * S1
M11 = sp.trigsimp((S1.T * f1)[0])

M = sp.Matrix([
    [M11, M12],
    [M12, M22]
])

Cq2, Sq2 = sp.symbols("C_q2 S_q2")

M_short = M.subs({
    sp.cos(q2): Cq2,
    sp.sin(q2): Sq2
})

from IPython.display import display, Math

display(Math(r"M(\boldsymbol{q}) = " + sp.latex(M_short)))

M_kane = sp.Matrix([
    [
        I1zz + (I2xx + m2*r2G**2)*sp.sin(q2)**2
        + I2zz*sp.cos(q2)**2,
        0
    ],
    [0, I2yy + m2*r2G**2]
])

diff = (M - M_kane).applyfunc(sp.trigsimp)

assert diff == sp.zeros(2)
print("CRBA and Kane mass matrices match.")


<IPython.core.display.Math object>

CRBA and Kane mass matrices match.


2 Link Arm の質量行列は以下である。それぞれの要素が一致するため、CRBAによる計算は正しいといえる。

$$
\begin{bmatrix}
I_{1zz} +  S_{q_2}^2 (I_{2xx} + m_2r_{2G}^2) + C_{q_2}^2 I_{2zz} & 0 \\
0 & m_2r_{2G}^2 + I_{2yy}
\end{bmatrix}
$$

## 6. 注意

CRBAの計算は、Tree構造の多体系について、一般化座標に対するの質量行列$M(q)$の計算をアルゴリズムである。

一方、接触条件などを導入した場合、運動方程式は、例えば次のように表せる。

$$
\begin{bmatrix}
M(q) & -J_c(q)^T \\
J_c(q) & 0 
\end{bmatrix}
\begin{bmatrix} \ddot{q} \\ \lambda \end{bmatrix} =
\begin{bmatrix}
\tau - h(q, \dot{q}) \\
-\dot{J}_c(q, \dot{q})\dot{q}
\end{bmatrix}
$$

ここでCRBAで計算できるのは、左上の$M(q)$であり、拘束ヤコビアン$J_c$や拘束力$\lambda$まで計算するわけではない。これらを綱領するには、追加の計算が必要になる。

以上